# Chapter 5: Transaction Processing and Recovery
*From: Database Internals*

## TL;DR

This chapter is about the **three cooperating subsystems** a storage engine needs to turn raw on-disk data structures into something that can honour the ACID contract under concurrent load and crashes:

- The **page cache** (buffer pool) stages dirty pages in memory, amortises I/O, and decides which pages to evict — a pure performance component that secretly constrains every durability and recovery choice below it.
- The **write-ahead log** converts durability into a sequential-write problem: every state change is logged (with an LSN) *before* the corresponding page can be flushed, which is what lets the page cache defer writes without losing them.
- The **concurrency-control subsystem** (locks + latches + OCC/MVCC/PCC) lets many transactions overlap without producing anomalies — but only at the isolation level the system is willing to pay for.
- These components are **tightly coupled through steal/force policies**. ARIES — the canonical recovery algorithm — picks `steal + no-force` and compensates with physical redo + logical undo and 3-phase recovery (analysis -> redo -> undo), which is strictly more permissive (and more work) than the alternatives.
- Two independent sets of synchronisation primitives coexist: **locks** (logical, key-based, held for the transaction, deadlock-prone) and **latches** (physical, page-based, short-lived, programmer-managed). Conflating them is the single most common source of confusion in this chapter.
- The running tension throughout the chapter: **efficiency wants to delay work and batch it; correctness wants to force it out before anything visible happens**. Every design choice — eviction policy, steal/force, OCC vs PCC, crabbing vs Blink-trees — is a specific answer to that tension.

## Chapter Roadmap

| # | Concept | One-line summary |
|---|---------|------------------|
| 1 | Page Cache and Buffer Management | A DB-managed in-memory staging layer that buffers dirty pages, serves cached reads, pins hot pages, and coordinates flushes with the WAL. |
| 2 | Page Replacement Algorithms | Strategies (FIFO, LRU family, CLOCK-sweep, TinyLFU) for deciding which cached page to evict when the pool is full. |
| 3 | Write-Ahead Log and Recovery | An append-only LSN-ordered operation log forced to disk before page modifications, enabling crash recovery and log trimming via checkpoints. |
| 4 | Steal/Force Policies and ARIES | How steal/no-steal and force/no-force determine whether undo/redo are required, and how ARIES uses steal/no-force with 3-phase recovery. |
| 5 | Isolation Levels and Anomalies | The ladder of SQL isolation levels and the read/write anomalies (dirty, nonrepeatable, phantom, lost update, write skew) each permits. |
| 6 | Concurrency Control: OCC, MVCC, PCC | Three families for coordinating overlapping transactions, differing in conflict assumptions and blocking behaviour. |
| 7 | Locks, Latches, and B-Tree Concurrency | Transaction-level logical locks vs storage-level physical latches; RW locks, latch crabbing, latch upgrading, and Blink-trees. |

---

## 1. Page Cache and Buffer Management

The storage engine sits on a two-level memory hierarchy — fast RAM, slow disk — and the **page cache** (also called the *buffer pool*, though the book prefers "page cache" because the role is caching, not pooling empty buffers) is the component that makes that hierarchy usable. It caches pages read from disk, accumulates writes against their cached copies, and decides when (and in what order) dirty pages are flushed back. It is effectively an application-specific reimplementation of the kernel page cache, which is precisely why many databases open files with `O_DIRECT`: they want full control over caching and readahead decisions, not the kernel's generic heuristics.

Three states matter for a cached page. A page is **paged in** when it is loaded from disk into a free slot; it is **dirty** when it has been modified but not yet flushed; and it is **referenced** (or pinned) when some caller is currently using it and must not have the slot reclaimed out from under them. Pinning is a deliberate optimisation: upper B-tree levels are narrow but hit on every query, so pinning the root and inner nodes means most queries only go to disk for the leaf levels, reducing effective tree height from `h` to `h - pinned_levels`.

The cache also gives the engine fine-grained control over prefetch and immediate eviction. Range scans can prefetch the next leaf before it's needed; maintenance jobs that touch pages once can hint that those pages should be evicted straight away (PostgreSQL uses a ring buffer for large sequential scans precisely for this reason). The page cache is the single point where the engine negotiates a trade-off between five competing objectives at once: postpone flushes, preemptively flush, flush in optimal order, stay within memory bounds, and don't lose data. The rest of this chapter is mostly a catalogue of techniques for balancing those five.

```mermaid
graph TB
    subgraph Logical[Logical B-Tree]
        P1[Page 1: root]
        P2[Page 2: inner]
        P3[Page 3: leaf]
        P4[Page 4: leaf]
    end

    subgraph Cache[Page Cache in RAM]
        S1[Slot A: Page 3 dirty]
        S2[Slot B: Page 1 clean pinned]
        S3[Slot C: Page 7 clean]
        S4[Slot D: free]
    end

    subgraph Disk[On-Disk File]
        D1[Page 1]
        D2[Page 2]
        D3[Page 3 stale]
        D4[Page 4]
    end

    Logical -.logical addr.-> Cache
    Cache -.flush on evict / checkpoint.-> Disk
    Disk -.page in on miss.-> Cache
```

### Page states and what the cache owes each

| State | Meaning | Can be evicted? | Flush required? |
|-------|---------|-----------------|-----------------|
| Clean + unreferenced | In sync with disk, nobody using it | Yes, immediately | No |
| Dirty + unreferenced | Modified in RAM, not on disk | Only after flush | Yes |
| Referenced (pinned) | In use by a thread | No | N/A until unpinned |
| Dirty + pinned | Being written right now | No | Yes, eventually |

### Where the trade-off lives

| Pressure | Wants what? | Opposed by |
|----------|-------------|------------|
| Throughput | Postpone flushes, batch them | Durability, recovery time |
| Eviction speed | Keep many clean pages ready | Throughput (flushing eagerly costs I/O) |
| Memory bound | Evict aggressively | Hit rate |
| Durability | Force flush on commit | Commit latency |
| Recovery time | Frequent checkpoints, shallow dirty-list | Throughput |

---

## 2. Page Replacement Algorithms

When the cache is full and a new page must be paged in, the eviction policy chooses the victim. The fantasy algorithm ("Bélády's optimal") evicts the page whose next access is farthest in the future, but that requires a crystal ball; real systems use heuristics that approximate it from observed access patterns. Crucially, **a bigger cache does not automatically evict less** — Bélády's anomaly shows that some naive algorithms (notably FIFO) can have *more* evictions with *more* slots. The algorithm matters.

The family tree is roughly: **FIFO** evicts in insertion order (pathologically bad for B-tree roots, which are paged in first and evicted first despite being hottest). **LRU** promotes a page back to the tail on each access, so recency is captured — but moving list nodes on every access is a concurrency bottleneck. **2Q** and **LRU-K** refine LRU by distinguishing hot from newly-seen pages so a single burst of scans doesn't pollute the cache with one-shot pages. **CLOCK-sweep** approximates LRU with just one access bit per page in a circular buffer, updated with compare-and-swap — cheap, concurrency-friendly, used by Linux and many databases. **TinyLFU** (the Caffeine library, and increasingly databases) flips the question from *which to evict* to *which to admit*, using a compact frequency histogram and a three-queue admission/probation/protected structure so that frequency — not just recency — drives retention.

```mermaid
graph LR
    subgraph Clock[CLOCK-sweep circular buffer]
        A[P1 bit=1] --> B[P2 bit=0]
        B --> C[P3 bit=1 pinned]
        C --> D[P4 bit=0]
        D --> E[P5 bit=1]
        E --> A
        Hand[clock hand] -.points at.-> B
    end
```

```mermaid
graph LR
    New[New page arrives] --> Adm[Admission queue LRU]
    Adm -->|frequency > victim?| Prob[Probation queue]
    Adm -->|else| Evict1[Evict immediately]
    Prob -->|accessed again| Prot[Protected queue]
    Prot -->|full, demote LRU| Prob
    Prob -->|not accessed| Evict2[Evict]
```

### Algorithms head-to-head

| Algorithm | Signal used | Data structure | Concurrency cost | Weakness |
|-----------|-------------|----------------|------------------|----------|
| FIFO | Insertion order | Queue | Low | Evicts hot roots; Belady's anomaly |
| LRU | Recency | Doubly-linked list + hash | High (relink on every access) | Scan pollution |
| 2Q / LRU-K | Recency + "seen before" | Two queues | Medium | Parameters to tune |
| CLOCK-sweep | Access bit (approx LRU) | Circular buffer | Very low (CAS) | Approximation, not exact LRU |
| TinyLFU | Frequency via sketch | Admission + probation + protected | Low (sketch + queues) | Cold-start, histogram memory |

> **Rule of thumb from the chapter:** under heavy load, recency alone is a weak predictor because everything looks "recent." Frequency-based or hybrid policies usually dominate pure LRU once the working set exceeds the cache.

---

## 3. Write-Ahead Log and Recovery

The WAL is the quiet hero of the chapter. It is an **append-only**, **sequential-write**, **LSN-ordered** disk-resident log whose single job is: *no page modification becomes durable on the main data files until the corresponding log record is already on disk*. That one rule is what lets the page cache defer dirty-page flushes arbitrarily far without sacrificing durability — the log is the durable copy of history; the data files are a lazy materialised view of it.

Each log record carries a monotonically increasing **log sequence number** (LSN). Records are collected in a **log buffer** in memory and emitted to disk by a **force** operation, which is triggered when the buffer fills, when the transaction manager needs to commit, or when the page cache needs to flush a dirty page (whose LSN of last modification must already be on disk). A transaction is considered committed only once the log has been forced up to its commit record's LSN — the commit is the log entry, not the page write.

Log content comes in three flavours: **physical** (before/after images of whole pages — fast redo, bulky), **logical** (an operation description like "insert key Y with value X" — compact but requires a consistent state to apply), or **combined** (the ARIES choice: physical redo for speed, logical undo for concurrency). Beyond operation records, the WAL also carries transaction control records (begin, commit, abort), **compensation log records (CLRs)** written during undo so that recovery-of-recovery doesn't re-undo the same work, and **checkpoint records** that let the log be trimmed.

Checkpoints are the log's escape valve. A **sync checkpoint** forces every dirty page and truncates the log behind it — conceptually simple but has to pause writers. A **fuzzy checkpoint** instead writes a `begin_checkpoint` record, lets flushing happen asynchronously, and writes `end_checkpoint` (with the dirty-page table and active-transaction table) when it's done. The `last_checkpoint` pointer in the log header advances to the `begin_checkpoint` LSN only once all its pages are on disk. Recovery starts from `last_checkpoint` — not the start of time — which is what keeps recovery bounded.

```mermaid
graph LR
    TXN[Transaction op] --> LB[Log buffer in RAM]
    LB -->|force: buffer full / commit / page flush| WAL[(WAL on disk)]
    WAL --> CKPT[Checkpoint process]
    CKPT -->|flush dirty pages| DATA[(Data files)]
    CKPT -->|advance last_checkpoint| TRIM[Trim old log segments]
    Crash((Crash)) -.restart.-> REC[Recovery]
    WAL -.replay from last_checkpoint.-> REC
    DATA -.starting state.-> REC
```

### Physical vs logical logging

| Aspect | Physical log | Logical log | Combined (ARIES) |
|--------|--------------|-------------|------------------|
| Record content | Before/after page images (or byte diffs) | Operation + inverse operation | Physical redo + logical undo |
| Redo speed | Fast (just overwrite) | Slower (re-run op) | Fast |
| Undo under concurrency | Hard (page may have changed) | Easy (op applies to current state) | Easy |
| Record size | Large | Small | Mixed |
| State required to apply | Exact page match | Consistent state | Mixed |

---

## 4. Steal/Force Policies and ARIES

Steal and force are the two binary choices that determine what the WAL actually has to carry. They are nominally about the page cache but are really about recovery — every combination maps directly onto whether undo records, redo records, or both must be logged.

**Steal** asks: can the page cache flush a dirty page belonging to an *uncommitted* transaction to make room for something else? If yes (**steal**), then disk may contain uncommitted changes, which means recovery must be able to **undo** them — so undo records must be in the log before a dirty page is stolen. If no (**no-steal**), uncommitted changes never leak to disk, so undo is not needed — but the cache must be large enough to hold every uncommitted transaction's dirty set. **Force** asks: must every page modified by a transaction be flushed *before* that transaction commits? If yes (**force**), committed changes are guaranteed on disk, so **redo** is not needed — but commit latency includes forcing every dirty page. If no (**no-force**), some committed changes may still be in cache at crash time, so recovery must **redo** them from the log.

The four corners of the matrix exist in real systems, but the interesting corner is **steal + no-force**, which requires *both* undo and redo logging and is strictly more work — in exchange for full freedom in flush scheduling and fast commits. This is what **ARIES** (Algorithm for Recovery and Isolation Exploiting Semantics, Mohan et al. 1992) picks, and it handles the complexity with a three-phase recovery that runs on restart:

1. **Analysis.** Scan forward from `last_checkpoint`. Rebuild the dirty page table (which pages may have uncommitted modifications) and the active transaction table (which transactions were in-flight at crash time). This determines where redo must start and which transactions must be undone.
2. **Redo.** From the earliest dirty-page LSN, **repeat history** — replay every logged change, whether from committed or uncommitted transactions, because the goal of this phase is just to reconstruct the exact pre-crash state. Physical redo makes this cheap.
3. **Undo.** Walk the uncommitted transactions backward in reverse chronological order, applying logical undo to each. Each undo emits a **compensation log record (CLR)** that describes the undo itself so that if the system crashes *again* mid-recovery, the next run sees the CLRs and doesn't re-undo.

The outcome: commits are cheap (no force), flush scheduling is unconstrained (steal is allowed), and recovery is bounded by the last checkpoint instead of the start of time.

```mermaid
graph TB
    Crash((Crash)) --> Restart[Restart]
    Restart --> A[Analysis phase<br/>Scan from last_checkpoint<br/>Rebuild dirty page table<br/>Rebuild active txn table]
    A --> R[Redo phase<br/>Repeat history from earliest<br/>dirty-page recLSN<br/>Apply physical redo for<br/>committed AND uncommitted]
    R --> U[Undo phase<br/>Walk uncommitted txns backward<br/>Apply logical undo<br/>Emit CLRs for each undo]
    U --> OK[Consistent state restored]
```

### Steal x Force matrix: what must be logged?

| | **Force** (flush before commit) | **No-Force** (lazy commit) |
|---|---|---|
| **No-Steal** (no uncommitted on disk) | Undo: no / Redo: no<br/>Simplest, slowest commits, huge cache needed | Undo: no / Redo: **yes**<br/>Deferred-update systems |
| **Steal** (uncommitted may be on disk) | Undo: **yes** / Redo: no<br/>Commit is expensive but recovery just needs undo | Undo: **yes** / Redo: **yes**<br/>**ARIES** — max freedom, max log content |

---

## 5. Isolation Levels and Anomalies

Isolation is the "I" in ACID, and in practice nobody runs full isolation — it's too expensive. Instead, databases offer a ladder of weaker isolation levels, and the ladder is defined by which **anomalies** each level allows. The SQL standard names three **read anomalies** and the chapter adds three **write anomalies**:

- **Dirty read:** T2 reads a value T1 wrote but hasn't committed yet; T1 then aborts, so T2 read a value that never existed.
- **Nonrepeatable (fuzzy) read:** T1 reads row R, T2 updates R and commits, T1 reads R again and sees a different value within the same transaction.
- **Phantom read:** like nonrepeatable, but for a *range* query — T1's second range read returns a different set of rows because T2 inserted or deleted matching rows.
- **Lost update:** T1 and T2 both read V, both write V based on their reads, and commit; whichever commits second silently overwrites the other.
- **Dirty write:** T2 overwrites a value T1 wrote but hasn't committed, so T2's result is based on state that may later be rolled back.
- **Write skew:** each transaction individually preserves an invariant, but their *combination* violates it (the classic two-accounts-sum-nonnegative example, where T1 and T2 each withdraw from different accounts while both read the combined balance).

The isolation ladder is then: **read uncommitted** allows everything; **read committed** forbids dirty reads; **repeatable read** also forbids nonrepeatable reads; **serializable** forbids everything. **Snapshot isolation** is a popular sibling of repeatable read: each transaction reads from a consistent snapshot taken at start time, and only one of two transactions writing the same key can commit — this rules out lost update but still permits write skew, which is why serializable is genuinely stronger than snapshot isolation despite often being conflated with it.

Serializability is a property of *some* equivalent serial order — it does not mandate a specific one. Two transactions with disjoint read/write sets can always run concurrently since any order is an equivalent serial order.

```mermaid
graph LR
    T1[T1: write V=10] -.visible before commit.-> T2[T2: read V=10]
    T1 -->|abort| X[rollback]
    T2 --> DR[Dirty read]

    A1[T1: read R=5] --> A2[T2: update R=7, commit]
    A2 --> A3[T1: read R=7]
    A3 --> NR[Nonrepeatable read]

    B1[A1=100, A2=150, constraint A1+A2 >= 0] --> B2[T1: withdraw 200 from A1]
    B1 --> B3[T2: withdraw 200 from A2]
    B2 --> B4[commit, A1 = -100]
    B3 --> B5[commit, A2 = -50]
    B4 --> WS[Write skew: invariant violated]
    B5 --> WS
```

### Isolation level x anomaly matrix

| Isolation level | Dirty read | Nonrepeatable read | Phantom read | Lost update | Write skew |
|-----------------|:----------:|:------------------:|:------------:|:-----------:|:----------:|
| Read uncommitted | allowed | allowed | allowed | allowed | allowed |
| Read committed | prevented | allowed | allowed | allowed | allowed |
| Repeatable read | prevented | prevented | allowed | allowed (implementation-dep.) | allowed |
| Snapshot isolation | prevented | prevented | prevented | prevented | **allowed** |
| Serializable | prevented | prevented | prevented | prevented | prevented |

---

## 6. Concurrency Control: OCC, MVCC, and PCC

There are three families of concurrency control, and the choice among them is really a bet on how often transactions actually conflict. If conflicts are rare, assume no conflict and validate afterward (**OCC**). If reads vastly outnumber writes and you want readers never to block, keep multiple versions around (**MVCC**). If conflicts are common, prevent them from happening by locking up front (**PCC**, which includes both lock-based and nonlocking timestamp-ordering variants).

**Optimistic concurrency control** splits each transaction into three phases: a **read phase** against a private context (tracking read set and write set), a **validation phase** that checks the accumulated sets against the read/write sets of concurrent transactions (backward-oriented checks already-committed transactions; forward-oriented checks in-progress ones), and a **write phase** that atomically installs the private writes. Validation and write phases together are a short critical section. OCC's win is that read phases never block; its weakness is that under contention, validation fails often, and retries are pure waste.

**Multiversion concurrency control** keeps multiple timestamped versions of each record so that reads can see a consistent snapshot without coordinating with writers. The transaction manager aims for at most one *uncommitted* version at a time; readers at snapshot time T see the latest version committed before T. MVCC is the standard implementation technique for snapshot isolation and is used by PostgreSQL, Oracle, and many others. It can be combined with locking (two-phase locking over versions) or with timestamp ordering.

**Pessimistic concurrency control** assumes conflict and prevents it. The lockless variant is **timestamp ordering**: each transaction gets a timestamp, each value tracks `max_read_timestamp` and `max_write_timestamp`, and an operation whose timestamp violates the tracked maxima is aborted (with the Thomas Write Rule saying outdated writes can simply be ignored). The lock-based variant is **two-phase locking (2PL)**: transactions acquire locks during a **growing phase**, then release them during a **shrinking phase**, with the rule that no lock is acquired after any has been released. 2PL is the classical path to serializability. It introduces deadlocks, which are handled either by detection (a waits-for graph cycle check, aborting the youngest transaction) or by prevention protocols like **wait-die** (only older transactions wait) or **wound-wait** (older transactions abort younger ones).

```mermaid
graph LR
    subgraph OCC[OCC lifecycle]
        R[Read phase<br/>private context<br/>track read/write sets] --> V[Validation phase<br/>check overlap with<br/>concurrent txns]
        V -->|conflict| Abort[Abort + retry]
        V -->|clean| W[Write phase<br/>install atomically]
    end
```

```mermaid
graph LR
    subgraph TPL[Two-Phase Locking]
        G[Growing phase<br/>acquire locks only] --> Peak[Lock peak]
        Peak --> S[Shrinking phase<br/>release locks only]
        S --> Commit[Commit]
    end
```

### OCC vs MVCC vs PCC

| Dimension | OCC | MVCC | PCC (2PL / timestamp ordering) |
|-----------|-----|------|-------------------------------|
| Conflict assumption | Rare | Readers/writers overlap, few write-write | Frequent |
| Do reads block writes? | No | No | Yes (2PL with exclusive locks) |
| Do writes block reads? | No | No (readers see older version) | Yes (2PL) |
| Wasted work on conflict | Abort + full restart of read phase | Abort of losing writer only | Blocking wait or deadlock abort |
| Space cost | Private contexts during read phase | Multiple versions per record | Lock table |
| Native isolation level | Serializable | Snapshot (usually) | Serializable |
| Deadlocks possible? | No (validation-based) | Only with lock-based MVCC | **Yes** — need detection or prevention |
| Best for | Read-mostly, low conflict | Read-heavy, long read txns | High-conflict OLTP |

---

## 7. Locks, Latches, and B-Tree Concurrency

The single most important distinction in this section: **locks** and **latches** are not the same thing, despite the systems-programming world using "lock" for both. A **lock** protects *logical* data integrity — it is held on a database key (or key range) for the duration of a transaction, participates in deadlock detection, and is managed by the lock manager outside the storage structure. A **latch** protects *physical* data integrity — it is held on a page (or part of a tree), is acquired and released inside a single operation, has no deadlock detection (the programmer must avoid deadlocks by discipline), and is what concurrent B-tree traversal actually uses. Even lockless concurrency control (OCC, timestamp ordering) still needs latches to protect page contents during reads, writes, splits, and merges.

The naive latching strategy — hold every latch from root to leaf for the whole operation — serialises the entire tree at the root and is unusable. The canonical optimisation is **latch crabbing** (also called latch coupling): descend the tree acquiring write latches, and at each level, release the parent's latch as soon as you know the operation cannot propagate up to it. For an insert that means: if the child is not full, the parent can't split, so release the parent. For a delete: if the child is not about to underflow, release the parent. Most operations never propagate up, so most operations end up holding only the target leaf's latch.

A refinement is **latch upgrading**: start with shared latches on the descent (cheap, highly concurrent), and only upgrade to exclusive at the leaf. If the leaf turns out to need a split/merge that propagates upward, re-walk and upgrade. The root is the last node to split (every child has to fill up first), so the common case is "shared latch at the root, never upgraded" — the root is latched *optimistically*, and the re-walk cost (pointer chasing) is rarely paid.

**Blink-trees** go further: every node has a **high key** and a **sibling pointer**, allowing a **half-split** state where a new right sibling is already linked via the sibling pointer before the parent knows about it. Lookups that descend toward a key and find the key exceeds the node's high key know the tree has been concurrently split and follow the sibling pointer forward — no need to hold the parent's latch during the descent, no need to abort on concurrent splits. This makes reads fully concurrent with structural modifications and eliminates a whole class of deadlocks.

```mermaid
graph TB
    subgraph Crab[Latch crabbing on insert]
        R[Root: acquire W-latch] --> C1{Child full?}
        C1 -->|no| REL1[Release root latch]
        REL1 --> L2[Inner: acquire W-latch]
        L2 --> C2{Child full?}
        C2 -->|no| REL2[Release inner latch]
        REL2 --> Leaf[Leaf: acquire W-latch, insert]
        C1 -->|yes| KEEP1[Keep root latch]
        C2 -->|yes| KEEP2[Keep inner latch]
    end
```

```mermaid
graph LR
    subgraph Blink[Blink-tree half-split]
        Par[Parent] -->|child ptr| Old[Node N]
        Old -->|sibling ptr| New[New sibling N']
        Lookup[Concurrent lookup] -->|key > N.high_key| Old
        Old -.follow sibling.-> New
        Par -.parent pointer update lazy.-> New
    end
```

### Locks vs latches

| Dimension | Lock | Latch |
|-----------|------|-------|
| Protects | Logical data (keys, key ranges) | Physical data (page contents, tree structure) |
| Scope | Whole transaction | Single page access or structural op |
| Duration | Until transaction commits/aborts | Microseconds to milliseconds |
| Granularity | Key / range | Page (sometimes sub-page) |
| Deadlock handling | Detect (waits-for graph) or prevent (wait-die, wound-wait) | Programmer discipline; none at runtime |
| Managed by | Lock manager | Storage engine / access method |
| Used by lockless CC? | No | **Yes** (still needed) |

### B-tree concurrency techniques

| Technique | Idea | Pays off because... |
|-----------|------|---------------------|
| Crab (coupling) | Release parent when child op won't propagate | Propagation to parent is rare |
| Latch upgrade | Shared on descent, upgrade at leaf | Most ops are non-structural |
| Optimistic root | Descend without expecting root split | Root splits are the rarest event in a tree |
| Blink-tree | High key + sibling pointer, allow half-split | Lookups follow sibling; no parent latch during descent |

---

## Back-of-Envelope: WAL throughput and checkpoint cadence

Given an OLTP workload, this estimation walks through whether the WAL disk can keep up, how fast the log buffer should be forced, and how often checkpoints must run to keep crash-recovery time bounded. The "big tension" from the TL;DR is visible directly in the numbers: more transactions per second means more WAL bandwidth *and* more dirty pages to checkpoint, so one sizing pressures the other.

In [ ]:
# Stdlib-only WAL + checkpoint sizing for a steady-state OLTP workload.

# --- Workload ---
txn_per_sec            = 20_000           # committed transactions per second
log_records_per_txn    = 4                # avg WAL records per transaction
avg_log_record_bytes   = 128              # physical-redo + logical-undo, typical
commit_sync            = True             # group-commit forces on commit boundary
group_commit_batch     = 200              # txns batched per fsync

# --- Hardware ---
wal_disk_mb_per_sec    = 500              # sequential write bandwidth (NVMe-ish)
wal_disk_iops_sync     = 10_000           # fsync ops/sec the device can sustain
page_size_bytes        = 8_192            # standard DB page
buffer_pool_pages      = 1_000_000        # 8 GiB buffer pool
dirty_page_fraction    = 0.20             # steady-state fraction that's dirty

# --- Recovery-time target ---
target_recovery_seconds = 30              # max acceptable restart time

# --- WAL throughput ---
log_bytes_per_sec = txn_per_sec * log_records_per_txn * avg_log_record_bytes
log_mb_per_sec    = log_bytes_per_sec / (1024 * 1024)
bw_utilization    = log_mb_per_sec / wal_disk_mb_per_sec

# --- fsync pressure (group commit) ---
fsync_per_sec     = txn_per_sec / group_commit_batch if commit_sync else 0
fsync_headroom    = wal_disk_iops_sync / fsync_per_sec if fsync_per_sec else float('inf')

# --- Checkpoint cadence from recovery bound ---
# In ARIES-style recovery, redo must replay all log between last_checkpoint and crash,
# so recovery time roughly equals (log bytes since checkpoint) / (replay bandwidth).
# Assume replay runs at ~50% of raw log write bandwidth.
replay_mb_per_sec = wal_disk_mb_per_sec * 0.5
max_log_between_ckpts_mb = replay_mb_per_sec * target_recovery_seconds
checkpoint_interval_sec  = max_log_between_ckpts_mb / log_mb_per_sec

# --- Dirty-page flush pressure at each checkpoint ---
dirty_pages        = buffer_pool_pages * dirty_page_fraction
dirty_bytes        = dirty_pages * page_size_bytes
dirty_mb           = dirty_bytes / (1024 * 1024)
# Flush must finish well within the interval (say, half of it).
required_flush_mb_s = dirty_mb / (checkpoint_interval_sec / 2)

print("=== WAL throughput ===")
print(f"Log write rate:          {log_mb_per_sec:>8.1f} MB/s")
print(f"WAL disk bandwidth:      {wal_disk_mb_per_sec:>8.1f} MB/s")
print(f"Bandwidth utilization:   {bw_utilization*100:>8.1f} %")
print(f"fsync/sec (group commit):{fsync_per_sec:>8.1f}")
print(f"fsync headroom:          {fsync_headroom:>8.1f}x device limit")

print("\n=== Checkpoint cadence (recovery-bounded) ===")
print(f"Target recovery time:    {target_recovery_seconds:>8d} s")
print(f"Max log between ckpts:   {max_log_between_ckpts_mb:>8.0f} MB")
print(f"=> Checkpoint every:     {checkpoint_interval_sec:>8.1f} s")

print("\n=== Flush pressure per checkpoint ===")
print(f"Dirty pages:             {dirty_pages:>10,.0f}")
print(f"Dirty data:              {dirty_mb:>8.1f} MB")
print(f"Required flush rate:     {required_flush_mb_s:>8.1f} MB/s")
print(f"(must fit in half the interval to not overlap next checkpoint)")

# --- Verdict ---
print("\n=== Verdict ===")
if bw_utilization > 0.6:
    print("WAL disk is a bottleneck -- consider separate log device or larger records batching.")
elif fsync_headroom < 3:
    print("fsync is the bottleneck -- increase group_commit_batch or add battery-backed cache.")
elif required_flush_mb_s > wal_disk_mb_per_sec:
    print("Checkpoints cannot finish in time -- shrink buffer pool or raise recovery-time target.")
else:
    print("System is balanced at this workload. Watch fsync latency under burst.")

---

## Trade-offs and Alternatives

The chapter's three big decisions, side by side.

### Decision 1: Page-replacement policy

| Approach | Pros | Cons | When to use |
|----------|------|------|-------------|
| FIFO | Trivial to implement, lock-free | Evicts hot roots; Belady's anomaly | Only sequential-scan ring buffers (PostgreSQL does this for large scans) |
| LRU (plain) | Accurate recency tracking | Relink cost under concurrency; scan pollution | Small caches, single-threaded |
| CLOCK-sweep | Near-LRU, CAS-only, concurrency-friendly | Approximate; no frequency info | General-purpose buffer pool, default choice |
| TinyLFU | Frequency-aware, resists scan pollution | Sketch memory, cold-start | Read-heavy mixed-workload caches |

### Decision 2: Concurrency control family

| Approach | Pros | Cons | When to use |
|----------|------|------|-------------|
| OCC | Non-blocking reads, simple model | Wasted work under contention | Read-mostly, rare conflicts |
| MVCC | Readers don't block writers; natural snapshot isolation | Multiple versions = space + GC | Mixed OLTP with long readers |
| PCC (2PL) | Strong serializability, well understood | Deadlocks, lock-table contention | High-conflict OLTP needing strict serializability |
| Timestamp ordering | Lockless, deadlock-free | Aborts on order violation, needs max_ts tracking | Distributed or deterministic systems |

### Decision 3: Steal / Force policy

| Approach | Undo needed? | Redo needed? | Commit latency | Recovery complexity |
|----------|:-----------:|:-----------:|:---------------:|:-------------------:|
| No-Steal + Force | No | No | Highest (flush on commit) | Trivial |
| No-Steal + No-Force | No | Yes | Medium | Redo only |
| Steal + Force | Yes | No | High | Undo only |
| **Steal + No-Force (ARIES)** | **Yes** | **Yes** | **Low** | **Full 3-phase** |

---

## Key Takeaways

1. **The page cache is the coupling point.** Its eviction policy, its steal/no-steal rule, and its flush schedule all directly shape what the WAL has to carry and how long recovery takes. It cannot be designed in isolation.
2. **Write-ahead is the universal durability trick.** Every durable change appears as a sequential append to the log *before* any random write to data files. This converts durability from a random-I/O problem into a sequential-I/O problem.
3. **Steal/Force defines recovery cost.** `Steal + No-Force` is the most performant and most complex combination; ARIES is its canonical implementation, with 3-phase analysis/redo/undo and compensation log records for crash-during-recovery.
4. **Isolation levels are defined by anomalies, not by implementation.** Serializable forbids all; snapshot isolation forbids most but still permits write skew; lower levels trade correctness for throughput. Knowing which anomalies a level permits is more useful than knowing its name.
5. **OCC, MVCC, and PCC are bets on conflict rate.** OCC wins when conflicts are rare; MVCC wins when long-running readers coexist with writers; PCC (2PL) wins when conflicts are frequent and serializability is non-negotiable.
6. **Locks and latches are different mechanisms with different failure modes.** Locks are logical, transaction-scoped, deadlock-managed. Latches are physical, op-scoped, programmer-managed. Every concurrency-control family still needs latches.
7. **B-tree concurrency is almost entirely about not holding the root latch.** Crabbing releases parent latches early; latch upgrading uses shared latches on descent; Blink-trees eliminate the need to hold parent latches during splits altogether. Each step buys more read-write concurrency at the cost of protocol complexity.

## See Also

- [[01-page-cache-and-buffer-management]]
- [[02-page-replacement-algorithms]]
- [[03-write-ahead-log-and-recovery]]
- [[04-steal-force-policies-and-aries]]
- [[05-isolation-levels-and-anomalies]]
- [[06-concurrency-control-strategies]]
- [[07-locks-latches-and-btree-concurrency]]